# Векторизация текста и классификация тональности

Сравниваются несколько способов представления англоязычных отзывов
и несколько базовых моделей для задачи определения тональности.

## Импорты и зависимости

In [1]:
import os
import re
import time
import warnings
import numpy as np
import pandas as pd

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler


from pathlib import Path

def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").exists() and (candidate / "requirements.txt").exists():
            return candidate
    return current.parent if current.name == "notebooks" else current

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
FIGURES_DIR = REPO_ROOT / "figures" / "text_vectorization"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / 'nltk_data').mkdir(parents=True, exist_ok=True)
(DATA_DIR / 'hf_cache').mkdir(parents=True, exist_ok=True)

 

NLTK_DATA_DIR = DATA_DIR / "nltk_data"
HF_CACHE_DIR = DATA_DIR / "hf_cache"
nltk.data.path.insert(0, str(NLTK_DATA_DIR))
os.environ["HF_HOME"] = str(HF_CACHE_DIR)

warnings.filterwarnings('ignore')

resources = ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']
for resource in resources:
    try:
        nltk.data.find(f'tokenizers/{resource}')
    except LookupError:
        try:
            nltk.data.find(f'corpora/{resource}')
        except LookupError:
            nltk.download(resource, download_dir=str(NLTK_DATA_DIR), quiet=True)

print('Все зависимости загружены успешно.')

Все зависимости загружены успешно.


## Подготовка данных

Сначала пытаемся взять реальный датасет тональности IMDB через `datasets`.
Если он недоступен, пробуем `nltk.corpus.movie_reviews`.
Если загрузка недоступна, используем небольшой синтетический набор отзывов.

In [2]:
def generate_synthetic_imdb(n_samples=2000, random_state=42):
    """
    Генерация синтетического датасета для демонстрации.
    Позитивные отзывы содержат слова с позитивной окраской,
    негативные — с негативной.
    """
    np.random.seed(random_state)

    positive_phrases = [
        "this movie was absolutely brilliant and amazing",
        "fantastic film with outstanding performances by all actors",
        "i loved every minute of this wonderful masterpiece",
        "excellent direction and superb cinematography throughout",
        "a truly great and entertaining experience for everyone",
        "the best film i have seen in years incredible",
        "beautiful story with compelling and memorable characters",
        "highly recommend this spectacular and engaging movie",
        "perfect blend of drama and humor delightful film",
        "impressive visuals and a wonderful heartwarming storyline",
        "the acting was superb and the plot was gripping",
        "an extraordinary achievement in modern cinema loved it",
        "riveting from start to finish a true masterpiece",
        "the screenplay was brilliant and the cast was perfect",
        "deeply moving and beautifully crafted emotional journey",
    ]

    negative_phrases = [
        "this movie was absolutely terrible and disappointing",
        "awful film with poor performances and weak plot",
        "i hated every minute of this dreadful disaster",
        "terrible direction and horrible cinematography throughout",
        "a truly bad and boring experience for everyone",
        "the worst film i have seen in years dreadful",
        "ugly story with dull and forgettable characters",
        "would not recommend this horrible and tedious movie",
        "poor blend of drama and failed humor awful film",
        "cheap visuals and a dreadful boring storyline",
        "the acting was terrible and the plot was confusing",
        "a complete failure in modern cinema hated it",
        "boring from start to finish a true disappointment",
        "the screenplay was bad and the cast was miscast",
        "deeply frustrating and poorly crafted disappointing mess",
    ]

    fillers = [
        "the film", "this movie", "the story", "the director",
        "the main character", "the plot", "the ending", "the beginning",
        "the cast", "the screenplay",
    ]

    connectors = [
        "and also", "furthermore", "in addition", "moreover",
        "as well as", "not only that but", "on top of that",
    ]

    texts, labels = [], []
    half = n_samples // 2

    for _ in range(half):
        n_sentences = np.random.randint(2, 5)
        sentences = np.random.choice(positive_phrases, n_sentences, replace=True)
        filler = np.random.choice(fillers)
        connector = np.random.choice(connectors)
        review = f"{sentences[0]}. {connector} {filler} {sentences[1]}"
        if n_sentences > 2:
            review += f". {sentences[2]}"
        texts.append(review)
        labels.append(1)

    for _ in range(half):
        n_sentences = np.random.randint(2, 5)
        sentences = np.random.choice(negative_phrases, n_sentences, replace=True)
        filler = np.random.choice(fillers)
        connector = np.random.choice(connectors)
        review = f"{sentences[0]}. {connector} {filler} {sentences[1]}"
        if n_sentences > 2:
            review += f". {sentences[2]}"
        texts.append(review)
        labels.append(0)

    indices = np.random.permutation(len(texts))
    texts = [texts[i] for i in indices]
    labels = [labels[i] for i in indices]
    return texts, labels


texts, labels = None, None
dataset_source = None

try:
    from datasets import load_dataset

    print('Загрузка IMDB через HuggingFace datasets...')
    dataset = load_dataset('imdb', cache_dir=str(HF_CACHE_DIR))
    train_data = dataset['train'].shuffle(seed=42).select(range(5000))
    test_data = dataset['test'].shuffle(seed=42).select(range(2000))
    texts = list(train_data['text']) + list(test_data['text'])
    labels = list(train_data['label']) + list(test_data['label'])
    dataset_source = 'HuggingFace IMDB (реальный датасет тональности)'
    print(f'Датасет загружен: {len(texts)} примеров')
except Exception as e:
    print(f'HuggingFace IMDB недоступен: {e}')

if texts is None:
    try:
        from nltk.corpus import movie_reviews

        try:
            nltk.data.find('corpora/movie_reviews')
        except LookupError:
            nltk.download('movie_reviews', download_dir=str(NLTK_DATA_DIR), quiet=True)

        print('Загрузка NLTK movie_reviews...')
        fileids = movie_reviews.fileids()
        texts = [movie_reviews.raw(fileid) for fileid in fileids]
        labels = [1 if movie_reviews.categories(fileid)[0] == 'pos' else 0 for fileid in fileids]

        permutation = np.random.RandomState(42).permutation(len(texts))
        texts = [texts[i] for i in permutation]
        labels = [int(labels[i]) for i in permutation]

        dataset_source = 'NLTK movie_reviews (реальный датасет тональности)'
        print(f'Датасет загружен: {len(texts)} примеров')
    except Exception as e:
        print(f'NLTK movie_reviews недоступен: {e}')

if texts is None:
    print('Реальные датасеты недоступны. Генерируем синтетический sentiment dataset...')
    texts, labels = generate_synthetic_imdb(n_samples=2000)
    dataset_source = 'Синтетический IMDB-подобный датасет'
    print(f'Синтетический датасет создан: {len(texts)} примеров')

print(f'\nИсточник данных: {dataset_source}')
print(f'Всего примеров: {len(texts)}')
print(f'Распределение классов: {pd.Series(labels).value_counts().to_dict()}')

Загрузка IMDB через HuggingFace datasets...


HuggingFace IMDB недоступен: Invalid HF URI 'hf://datasets/imdb@e6281661ce1c48d982bc483cf8a173c1bbeb5d31/.huggingface.yaml'. Repository id must be 'namespace/name', got 'imdb'.
Загрузка NLTK movie_reviews...
Датасет загружен: 2000 примеров

Источник данных: NLTK movie_reviews (реальный датасет тональности)
Всего примеров: 2000
Распределение классов: {1: 1000, 0: 1000}


In [3]:
# Делим данные на обучающую и тестовую части
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f'Источник: {dataset_source}')
print(f'Обучающая выборка: {len(X_train_raw)} примеров')
print(f'Тестовая выборка:  {len(X_test_raw)} примеров')

print('\nПример позитивного отзыва:')
pos_indices = [i for i, label in enumerate(labels) if label == 1]
print(texts[pos_indices[0]][:300])

print('\nПример негативного отзыва:')
neg_indices = [i for i, label in enumerate(labels) if label == 0]
print(texts[neg_indices[0]][:300])

Источник: NLTK movie_reviews (реальный датасет тональности)
Обучающая выборка: 1600 примеров
Тестовая выборка:  400 примеров

Пример позитивного отзыва:
the verdict : spine-chilling drama from horror maestro stephen king , featuring an outstanding , oscar-winning performance from kathy bates . 
geez , french and saunders had a field day when they set to work on parodying this ! 
sorry , non-british readers may not be familiar with french and saunder

Пример негативного отзыва:
 " the 44 caliber killer has struck again . " 
starring john leguizamo , mira sorvino , adrian brody , jennifer esposito , michael rispoli , bebe neuwirth . 
rated r . 
summer of sam will be remembered as a waste of spike lee's abilities . 
lee is a great filmmaker , often exhibiting kinetic visual 


## Очистка и нормализация текста

Дальше идёт простой конвейер: очистка HTML и ссылок, токенизация,
удаление стоп-слов и лемматизация.

In [4]:
# Инициализация лемматизатора и стоп-слов
_lemmatizer = WordNetLemmatizer()
_stop_words = set(stopwords.words('english'))


def preprocess_text(text: str) -> str:
    """
    Предобработка текста:
    1. Приведение к нижнему регистру
    2. Удаление HTML-тегов и URL
    3. Удаление специальных символов (оставляем только буквы)
    4. Токенизация через NLTK
    5. Удаление стоп-слов
    6. Лемматизация через WordNetLemmatizer

    Параметры:
        text: исходная строка текста

    Возвращает:
        Предобработанная строка
    """
    # 1. Нижний регистр
    text = text.lower()

    # 2. Удаление HTML-тегов
    text = re.sub(r'<[^>]+>', ' ', text)

    # 3. Удаление URL
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # 4. Оставляем только буквы и пробелы
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 5. Нормализация пробелов
    text = re.sub(r'\s+', ' ', text).strip()

    # 6. Токенизация
    tokens = word_tokenize(text)

    # 7. Удаление стоп-слов и коротких токенов (< 2 символов)
    tokens = [
        t for t in tokens
        if t not in _stop_words and len(t) > 2
    ]

    # 8. Лемматизация
    tokens = [_lemmatizer.lemmatize(t) for t in tokens]

    return ' '.join(tokens)


# Тест функции
test_examples = [
    "This movie was absolutely BRILLIANT! The acting is amazing.",
    "<br>Terrible film, I hated every minutes of watching it...",
    "The dogs are running faster than the cats in this beautiful park!",
]

print('Примеры предобработки:')
print('=' * 60)
for example in test_examples:
    processed = preprocess_text(example)
    print(f'Оригинал:    {example}')
    print(f'Обработано:  {processed}')
    print('-' * 60)

Примеры предобработки:
Оригинал:    This movie was absolutely BRILLIANT! The acting is amazing.
Обработано:  movie absolutely brilliant acting amazing
------------------------------------------------------------
Оригинал:    <br>Terrible film, I hated every minutes of watching it...
Обработано:  terrible film hated every minute watching
------------------------------------------------------------
Оригинал:    The dogs are running faster than the cats in this beautiful park!
Обработано:  dog running faster cat beautiful park
------------------------------------------------------------


In [5]:
# Применяем предобработку ко всему датасету
print('Предобработка обучающей части...')
t0 = time.time()
X_train = [preprocess_text(text) for text in X_train_raw]
t1 = time.time()
print(f'Обучающая часть обработана за {t1 - t0:.1f} сек.')

print('Предобработка тестовой части...')
X_test = [preprocess_text(text) for text in X_test_raw]
t2 = time.time()
print(f'Тестовая часть обработана за {t2 - t1:.1f} сек.')

# Статистика по длинам текстов
train_lengths = [len(t.split()) for t in X_train]
print(f'\nСтатистика длин (в токенах) после предобработки:')
print(f'  Среднее:   {np.mean(train_lengths):.1f}')
print(f'  Медиана:   {np.median(train_lengths):.1f}')
print(f'  Мин/Макс:  {min(train_lengths)} / {max(train_lengths)}')

Предобработка обучающей части...
Обучающая часть обработана за 3.8 сек.
Предобработка тестовой части...
Тестовая часть обработана за 0.9 сек.

Статистика длин (в токенах) после предобработки:
  Среднее:   346.6
  Медиана:   324.0
  Мин/Макс:  6 / 1356


## TF-IDF и пространство признаков

Проверим две конфигурации:
- **Униграммы**: `ngram_range=(1, 1)`
- **Биграммы**: `ngram_range=(1, 2)`

In [6]:
# TF-IDF с униграммами
print('Строим TF-IDF по униграммам...')
tfidf_unigram = TfidfVectorizer(
    ngram_range=(1, 1),
    max_features=20000,    # ограничиваем словарь
    min_df=2,              # минимум 2 документа
    max_df=0.95,           # не более 95% документов
    sublinear_tf=True      # log(1 + tf) — сглаживание частот
)

X_train_uni = tfidf_unigram.fit_transform(X_train)
X_test_uni = tfidf_unigram.transform(X_test)

print(f'Матрица униграмм: {X_train_uni.shape}')
print(f'Размер словаря: {len(tfidf_unigram.vocabulary_)}')

# ТОП-20 слов по IDF (самые редкие = наиболее информативные)
feature_names = tfidf_unigram.get_feature_names_out()
idf_scores = tfidf_unigram.idf_
top20_idx = np.argsort(idf_scores)[-20:][::-1]
print(f'\nТОП-20 наиболее редких токенов (высокий IDF):')
print(', '.join([feature_names[i] for i in top20_idx]))

Строим TF-IDF по униграммам...
Матрица униграмм: (1600, 18916)
Размер словаря: 18916

ТОП-20 наиболее редких токенов (высокий IDF):
zwigoff, manufacture, mangled, manhunter, maniacal, maniacally, manifesting, manifold, manly, manni, manor, mantegna, mantle, marathon, mandoki, marcie, marcy, mariachi, mariah, marienbad


In [7]:
# TF-IDF с биграммами
print('Строим TF-IDF по биграммам...')
tfidf_bigram = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,    # больше фичей для биграмм
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_bi = tfidf_bigram.fit_transform(X_train)
X_test_bi = tfidf_bigram.transform(X_test)

print(f'Матрица биграмм: {X_train_bi.shape}')
print(f'Размер словаря: {len(tfidf_bigram.vocabulary_)}')

# Примеры биграмм
bigram_features = [f for f in tfidf_bigram.get_feature_names_out() if ' ' in f]
print(f'\nПримеры биграмм (первые 20):')
print(', '.join(bigram_features[:20]))

Строим TF-IDF по биграммам...
Матрица биграмм: (1600, 50000)
Размер словаря: 50000

Примеры биграмм (первые 20):
aaa team, aardman animation, aaron eckhart, aaron spelling, abandon farm, abandon much, abandon ship, abandoned building, abandoned church, abandoned house, abandoned warehouse, abby asks, abel ferrara, ability become, ability convey, ability create, ability entertain, ability fold, ability get, ability look


## Обучение моделей

Для сравнения берём три базовых варианта:
- **LogisticRegression**
- **LinearSVC**
- **MultinomialNB**

In [8]:
def train_and_evaluate(X_tr, X_te, y_tr, y_te, classifier, name):
    """
    Обучение классификатора и вычисление метрик.

    Параметры:
        X_tr: обучающая матрица признаков
        X_te: тестовая матрица признаков
        y_tr: метки для обучения
        y_te: метки для теста
        classifier: sklearn-совместимый классификатор
        name: подпись эксперимента

    Возвращает:
        dict с метриками
    """
    t0 = time.time()
    classifier.fit(X_tr, y_tr)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = classifier.predict(X_te)
    predict_time = time.time() - t0

    acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred, average='weighted')

    print(f'[{name}]')
    print(f'  Accuracy: {acc:.4f}  |  F1: {f1:.4f}')
    print(f'  Обучение: {train_time:.2f}с  |  Предсказание: {predict_time:.3f}с')

    return {
        'name': name,
        'accuracy': acc,
        'f1': f1,
        'train_time': train_time,
        'predict_time': predict_time,
    }


# Словарь классификаторов
classifiers = {
    'LogisticRegression': LogisticRegression(
        C=1.0,
        max_iter=1000,
        solver='lbfgs',
        random_state=42
    ),
    'LinearSVC': LinearSVC(
        C=1.0,
        max_iter=2000,
        random_state=42
    ),
    'MultinomialNB': MultinomialNB(
        alpha=0.1   # сглаживание Лапласа
    ),
}

print('Классификаторы инициализированы.')

Классификаторы инициализированы.


In [9]:
# Обучение на униграммах
print('=' * 60)
print('ЭКСПЕРИМЕНТ 1: TF-IDF на униграммах (1,1)')
print('=' * 60)

results_uni = []
for clf_name, clf in classifiers.items():
    # MultinomialNB требует неотрицательных значений (TF-IDF >= 0, всё ок)
    result = train_and_evaluate(
        X_train_uni, X_test_uni,
        y_train, y_test,
        clf,
        f'{clf_name} + Unigram'
    )
    result['vectorizer'] = 'TF-IDF Unigram'
    result['classifier'] = clf_name
    results_uni.append(result)
    print()

ЭКСПЕРИМЕНТ 1: TF-IDF на униграммах (1,1)
[LogisticRegression + Unigram]
  Accuracy: 0.8500  |  F1: 0.8500
  Обучение: 0.02с  |  Предсказание: 0.000с

[LinearSVC + Unigram]
  Accuracy: 0.8675  |  F1: 0.8675
  Обучение: 0.04с  |  Предсказание: 0.000с

[MultinomialNB + Unigram]
  Accuracy: 0.8100  |  F1: 0.8096
  Обучение: 0.00с  |  Предсказание: 0.000с



In [10]:
# Обучение на биграммах
print('=' * 60)
print('ЭКСПЕРИМЕНТ 2: TF-IDF на биграммах (1,2)')
print('=' * 60)

# Переинициализируем классификаторы для чистоты эксперимента
classifiers_bi = {
    'LogisticRegression': LogisticRegression(
        C=1.0, max_iter=1000, solver='lbfgs', random_state=42
    ),
    'LinearSVC': LinearSVC(
        C=1.0, max_iter=2000, random_state=42
    ),
    'MultinomialNB': MultinomialNB(alpha=0.1),
}

results_bi = []
for clf_name, clf in classifiers_bi.items():
    result = train_and_evaluate(
        X_train_bi, X_test_bi,
        y_train, y_test,
        clf,
        f'{clf_name} + Bigram'
    )
    result['vectorizer'] = 'TF-IDF Bigram'
    result['classifier'] = clf_name
    results_bi.append(result)
    print()

ЭКСПЕРИМЕНТ 2: TF-IDF на биграммах (1,2)
[LogisticRegression + Bigram]
  Accuracy: 0.8525  |  F1: 0.8525
  Обучение: 0.06с  |  Предсказание: 0.000с

[LinearSVC + Bigram]
  Accuracy: 0.8750  |  F1: 0.8750
  Обучение: 0.05с  |  Предсказание: 0.000с

[MultinomialNB + Bigram]
  Accuracy: 0.8225  |  F1: 0.8225
  Обучение: 0.00с  |  Предсказание: 0.001с



## Сравнение униграмм и биграмм

In [11]:
# Собираем таблицу: униграммы против биграмм
comparison_data = []
for r_uni, r_bi in zip(results_uni, results_bi):
    clf_name = r_uni['classifier']
    comparison_data.append({
        'Классификатор': clf_name,
        'Accuracy (униграммы)': f"{r_uni['accuracy']:.4f}",
        'Accuracy (биграммы)': f"{r_bi['accuracy']:.4f}",
        'F1 (униграммы)': f"{r_uni['f1']:.4f}",
        'F1 (биграммы)': f"{r_bi['f1']:.4f}",
        'Разница Accuracy': f"{r_bi['accuracy'] - r_uni['accuracy']:+.4f}",
        'Разница F1': f"{r_bi['f1'] - r_uni['f1']:+.4f}",
    })

df_comparison = pd.DataFrame(comparison_data)
print('Сравнение: униграммы против биграмм')
print('=' * 80)
print(df_comparison.to_string(index=False))
print()
print('Разница = Bigram - Unigram; положительное значение означает выигрыш биграмм.')

Сравнение: униграммы против биграмм
     Классификатор Accuracy (униграммы) Accuracy (биграммы) F1 (униграммы) F1 (биграммы) Разница Accuracy Разница F1
LogisticRegression               0.8500              0.8525         0.8500        0.8525          +0.0025    +0.0025
         LinearSVC               0.8675              0.8750         0.8675        0.8750          +0.0075    +0.0075
     MultinomialNB               0.8100              0.8225         0.8096        0.8225          +0.0125    +0.0129

Разница = Bigram - Unigram; положительное значение означает выигрыш биграмм.


In [12]:
# Подробный classification_report для лучшей конфигурации
all_results = results_uni + results_bi
best_result = max(all_results, key=lambda x: x['f1'])
print(f'Лучшая конфигурация по F1: {best_result["name"]}')
print(f'Accuracy: {best_result["accuracy"]:.4f}  |  F1: {best_result["f1"]:.4f}')

# Пересоздаём лучшую модель и выводим полный отчёт
if best_result['vectorizer'] == 'TF-IDF Unigram':
    X_tr_best, X_te_best = X_train_uni, X_test_uni
else:
    X_tr_best, X_te_best = X_train_bi, X_test_bi

best_clf_name = best_result['classifier']
clf_map = {
    'LogisticRegression': LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42),
    'LinearSVC': LinearSVC(C=1.0, max_iter=2000, random_state=42),
    'MultinomialNB': MultinomialNB(alpha=0.1),
}
best_clf = clf_map[best_clf_name]
best_clf.fit(X_tr_best, y_train)
y_pred_best = best_clf.predict(X_te_best)

print('\nПодробный отчёт classification_report:')
print(classification_report(y_test, y_pred_best, target_names=['Негатив', 'Позитив']))

Лучшая конфигурация по F1: LinearSVC + Bigram
Accuracy: 0.8750  |  F1: 0.8750

Подробный отчёт classification_report:
              precision    recall  f1-score   support

     Негатив       0.87      0.89      0.88       200
     Позитив       0.88      0.86      0.87       200

    accuracy                           0.88       400
   macro avg       0.88      0.88      0.87       400
weighted avg       0.88      0.88      0.87       400



## Проверка `HashingVectorizer`

`HashingVectorizer` удобен там, где не хочется хранить словарь
и отдельно вызывать `fit`. Цена за это известна: возможны коллизии,
а признаки потом уже не так просто интерпретировать.

In [13]:
# Feature Hashing с разными размерами пространства признаков
print('=' * 60)
print('ЭКСПЕРИМЕНТ 3: Feature Hashing (HashingVectorizer)')
print('=' * 60)

results_hashing = []

# Проверяем разные размеры хэш-пространства
for n_features in [2**14, 2**16, 2**18]:  # 16K, 64K, 256K
    print(f'\nHashingVectorizer с n_features={n_features} ({n_features // 1024}K):')

    hasher = HashingVectorizer(
        n_features=n_features,
        ngram_range=(1, 2),
        alternate_sign=False,  # для MultinomialNB нужны неотрицательные
        norm='l2'
    )

    t0 = time.time()
    # HashingVectorizer не требует fit — только transform
    X_train_hash = hasher.transform(X_train)
    X_test_hash = hasher.transform(X_test)
    hash_time = time.time() - t0
    print(f'  Векторизация: {hash_time:.2f}с')
    print(f'  Размер матрицы: {X_train_hash.shape}')

    # Обучаем LogisticRegression (LinearSVC нестабилен без нормализации в некоторых конфигурациях)
    for clf_name, clf in [
        ('LogisticRegression', LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42)),
        ('LinearSVC', LinearSVC(C=1.0, max_iter=2000, random_state=42)),
    ]:
        result = train_and_evaluate(
            X_train_hash, X_test_hash,
            y_train, y_test,
            clf,
            f'{clf_name} + Hashing({n_features // 1024}K)'
        )
        result['vectorizer'] = f'Hashing {n_features // 1024}K'
        result['classifier'] = clf_name
        result['n_features'] = n_features
        results_hashing.append(result)

print('\nHashing эксперименты завершены.')

ЭКСПЕРИМЕНТ 3: Feature Hashing (HashingVectorizer)

HashingVectorizer с n_features=16384 (16K):
  Векторизация: 0.50с
  Размер матрицы: (1600, 16384)
[LogisticRegression + Hashing(16K)]
  Accuracy: 0.7850  |  F1: 0.7850
  Обучение: 0.04с  |  Предсказание: 0.001с
[LinearSVC + Hashing(16K)]
  Accuracy: 0.8025  |  F1: 0.8025
  Обучение: 0.08с  |  Предсказание: 0.001с

HashingVectorizer с n_features=65536 (64K):
  Векторизация: 0.49с
  Размер матрицы: (1600, 65536)
[LogisticRegression + Hashing(64K)]
  Accuracy: 0.8075  |  F1: 0.8075
  Обучение: 0.06с  |  Предсказание: 0.001с
[LinearSVC + Hashing(64K)]
  Accuracy: 0.8575  |  F1: 0.8575
  Обучение: 0.08с  |  Предсказание: 0.001с

HashingVectorizer с n_features=262144 (256K):
  Векторизация: 0.49с
  Размер матрицы: (1600, 262144)
[LogisticRegression + Hashing(256K)]
  Accuracy: 0.8100  |  F1: 0.8100
  Обучение: 0.17с  |  Предсказание: 0.001с
[LinearSVC + Hashing(256K)]
  Accuracy: 0.8450  |  F1: 0.8450
  Обучение: 0.10с  |  Предсказание: 0.0

In [14]:
# Смотрим, как `HashingVectorizer` держится рядом с TF-IDF
print('Сравнение TF-IDF с биграммами и HashingVectorizer для LogisticRegression:')
print('-' * 70)

lr_bi = next(r for r in results_bi if r['classifier'] == 'LogisticRegression')
print(f"TF-IDF Bigram (50K фичей):  Accuracy={lr_bi['accuracy']:.4f}  F1={lr_bi['f1']:.4f}")

for r in results_hashing:
    if r['classifier'] == 'LogisticRegression':
        print(f"Hashing {r['n_features']//1024:>4}K:              Accuracy={r['accuracy']:.4f}  F1={r['f1']:.4f}")

print()
print('Короткий вывод: HashingVectorizer даёт сопоставимый результат')
print('и не требует отдельного словаря, что удобно для потоковой обработки.')

Сравнение TF-IDF с биграммами и HashingVectorizer для LogisticRegression:
----------------------------------------------------------------------
TF-IDF Bigram (50K фичей):  Accuracy=0.8525  F1=0.8525
Hashing   16K:              Accuracy=0.7850  F1=0.7850
Hashing   64K:              Accuracy=0.8075  F1=0.8075
Hashing  256K:              Accuracy=0.8100  F1=0.8100

Короткий вывод: HashingVectorizer даёт сопоставимый результат
и не требует отдельного словаря, что удобно для потоковой обработки.


## Сводка по всем конфигурациям

In [15]:
# Собираем все результаты в единую таблицу
all_experiments = results_uni + results_bi + results_hashing

summary_rows = []
for r in all_experiments:
    summary_rows.append({
        'Векторизация': r['vectorizer'],
        'Классификатор': r['classifier'],
        'Accuracy': round(r['accuracy'], 4),
        'F1 (weighted)': round(r['f1'], 4),
        'Время обучения, с': round(r['train_time'], 3),
    })

df_summary = pd.DataFrame(summary_rows)
df_summary = df_summary.sort_values('F1 (weighted)', ascending=False).reset_index(drop=True)

print('ИТОГОВАЯ ТАБЛИЦА СРАВНЕНИЯ ВСЕХ КОНФИГУРАЦИЙ')
print('=' * 80)
print(df_summary.to_string(index=True))

print()
print(f'Лучшая конфигурация (по F1):')
best_row = df_summary.iloc[0]
print(f"  Векторизация: {best_row['Векторизация']}")
print(f"  Классификатор: {best_row['Классификатор']}")
print(f"  Accuracy: {best_row['Accuracy']}")
print(f"  F1: {best_row['F1 (weighted)']}")

ИТОГОВАЯ ТАБЛИЦА СРАВНЕНИЯ ВСЕХ КОНФИГУРАЦИЙ
      Векторизация       Классификатор  Accuracy  F1 (weighted)  Время обучения, с
0    TF-IDF Bigram           LinearSVC    0.8750         0.8750              0.047
1   TF-IDF Unigram           LinearSVC    0.8675         0.8675              0.036
2      Hashing 64K           LinearSVC    0.8575         0.8575              0.084
3    TF-IDF Bigram  LogisticRegression    0.8525         0.8525              0.056
4   TF-IDF Unigram  LogisticRegression    0.8500         0.8500              0.021
5     Hashing 256K           LinearSVC    0.8450         0.8450              0.095
6    TF-IDF Bigram       MultinomialNB    0.8225         0.8225              0.003
7     Hashing 256K  LogisticRegression    0.8100         0.8100              0.169
8   TF-IDF Unigram       MultinomialNB    0.8100         0.8096              0.002
9      Hashing 64K  LogisticRegression    0.8075         0.8075              0.065
10     Hashing 16K           LinearSVC    

In [16]:
# Сводная статистика по методам векторизации
print('СРЕДНИЕ МЕТРИКИ ПО МЕТОДАМ ВЕКТОРИЗАЦИИ')
print('=' * 50)

# Группируем только по типу векторизации (Unigram / Bigram / Hashing)
df_summary['Тип векторизации'] = df_summary['Векторизация'].apply(
    lambda x: 'TF-IDF Unigram' if 'Unigram' in x
    else ('TF-IDF Bigram' if 'Bigram' in x else 'Hashing')
)

vec_stats = df_summary.groupby('Тип векторизации')[['Accuracy', 'F1 (weighted)']].agg(['mean', 'max'])
vec_stats.columns = ['Avg Accuracy', 'Max Accuracy', 'Avg F1', 'Max F1']
vec_stats = vec_stats.round(4)
print(vec_stats.to_string())

print()
print('СРЕДНИЕ МЕТРИКИ ПО КЛАССИФИКАТОРАМ')
print('=' * 50)

clf_stats = df_summary.groupby('Классификатор')[['Accuracy', 'F1 (weighted)']].agg(['mean', 'max'])
clf_stats.columns = ['Avg Accuracy', 'Max Accuracy', 'Avg F1', 'Max F1']
clf_stats = clf_stats.round(4)
print(clf_stats.to_string())

СРЕДНИЕ МЕТРИКИ ПО МЕТОДАМ ВЕКТОРИЗАЦИИ
                  Avg Accuracy  Max Accuracy  Avg F1  Max F1
Тип векторизации                                            
Hashing                 0.8179        0.8575  0.8179  0.8575
TF-IDF Bigram           0.8500        0.8750  0.8500  0.8750
TF-IDF Unigram          0.8425        0.8675  0.8424  0.8675

СРЕДНИЕ МЕТРИКИ ПО КЛАССИФИКАТОРАМ
                    Avg Accuracy  Max Accuracy  Avg F1  Max F1
Классификатор                                                 
LinearSVC                 0.8495        0.8750  0.8495  0.8750
LogisticRegression        0.8210        0.8525  0.8210  0.8525
MultinomialNB             0.8162        0.8225  0.8160  0.8225


In [17]:
# Небольшой итог
print('ИТОГ')
print('=' * 60)
print('1. TF-IDF на биграммах чаще даёт небольшой, но стабильный прирост,')
print('   потому что начинает видеть короткий контекст вроде "not good".')
print()
print('2. Среди базовых моделей здесь обычно спокойнее всего ведёт себя')
print('   LogisticRegression; LinearSVC идёт рядом, Naive Bayes быстрее,')
print('   но по качеству обычно попроще.')
print()
print('3. HashingVectorizer полезен, когда важны потоковая обработка и')
print('   фиксированный размер признакового пространства.')
print()
best_config = df_summary.iloc[0]
print('Лучший вариант в этом прогоне:')
print(f"  {best_config['Векторизация']} + {best_config['Классификатор']}")
print(f"  Accuracy: {best_config['Accuracy']}  |  F1: {best_config['F1 (weighted)']}")

ИТОГ
1. TF-IDF на биграммах чаще даёт небольшой, но стабильный прирост,
   потому что начинает видеть короткий контекст вроде "not good".

2. Среди базовых моделей здесь обычно спокойнее всего ведёт себя
   LogisticRegression; LinearSVC идёт рядом, Naive Bayes быстрее,
   но по качеству обычно попроще.

3. HashingVectorizer полезен, когда важны потоковая обработка и
   фиксированный размер признакового пространства.

Лучший вариант в этом прогоне:
  TF-IDF Bigram + LinearSVC
  Accuracy: 0.875  |  F1: 0.875
